In [1]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [2]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [3]:
dim_tipo_servicio = pd.read_sql_table('mensajeria_tiposervicio', co_sa)


# Transformations

In [4]:
dim_tipo_servicio.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           4 non-null      int64
 1   nombre       4 non-null      str  
 2   descripcion  4 non-null      str  
dtypes: int64(1), str(2)
memory usage: 228.0 bytes


In [ ]:

dim_tipo_servicio.rename(columns={'id': 'id_tipo_servicio','nombre': 'nombre_tipo_servicio'}, inplace=True)
dim_tipo_servicio.describe(include='all')

,id,nombre,descripcion
count,4.000000,4,4
unique,NaN,4,4
top,NaN,Administrativo,Administrativo
freq,NaN,1,1
mean,2.500000,NaN,NaN
std,1.290994,NaN,NaN
min,1.000000,NaN,NaN
25%,1.750000,NaN,NaN
50%,2.500000,NaN,NaN
75%,3.250000,NaN,NaN


In [6]:
dim_tipo_servicio.replace({np.nan: 'no aplica', ' ': 'no aplica','':'no_aplica'}, inplace=True)
#dim_tipo_servicio["saved"] = date.today()

,id,nombre,descripcion
0,1,Administrativo,Administrativo
1,2,Comercial,Comercial
2,3,Clínico,Clínico
3,4,Urgencia Vital,Urgencia Vital


# load

In [7]:
dim_tipo_servicio.head()

,id,nombre,descripcion
0,1,Administrativo,Administrativo
1,2,Comercial,Comercial
2,3,Clínico,Clínico
3,4,Urgencia Vital,Urgencia Vital


In [8]:
dim_tipo_servicio.to_sql('dim_tipo_servicio', etl_conn, if_exists='replace',index_label='key_dim_tipo_servicio')

4